# 🤖 Model Training & Evaluation — MViTv2 (Baseline Transformer)
Notebook ini adalah controller untuk fine-tune dan mengevaluasi **MViTv2** (torchvision,
pretrained Kinetics-400) sebagai baseline pembanding **CNNsa-BiLSTM** (proposed model),
mengikuti metodologi paper *Word-Level BISINDO: A Novel Video Indonesian Sign Language
Dataset and Baseline Methods*.

Beda dari CNNsa-BiLSTM: MViTv2 makan **raw video clip** (16 frame, 224x224 RGB) langsung
dari output PreProcessing — tidak butuh FeatureExtraction/keypoint sama sekali.


## 1. Setup & Cek Dataset

In [ ]:
import sys
import torch
from pathlib import Path

# Append root path (notebook di Training/MViTv2/ -> naik 2 level ke ROOT)
sys.path.insert(0, str(Path.cwd().parent.parent))

from Training.MViTv2.Config.mvit_config import verify_data_exists, create_output_dirs, TRAIN_CONFIG
from Training.Scripts.device_utils import get_device

# Pastikan frame_manifest.json (output PreProcessing) sudah ada
verify_data_exists()
create_output_dirs()

# get_device() print detail GPU (atau warning jelas kalau jatuh ke CPU)
device = get_device()

print(f"
Batch size: {TRAIN_CONFIG['batch_size']} (kecil karena VRAM terbatas + clip video besar)")


## 2. Model Summary (MViTv2 Baseline)
Lihat berapa persen parameter yang benar-benar di-fine-tune (sisanya dibekukan dari bobot Kinetics-400).

In [ ]:
from Training.MViTv2.Models.mvit_v2 import build_mvit_v2
from Training.MViTv2.Config.mvit_config import MODEL_CONFIG

model = build_mvit_v2(
    num_classes=MODEL_CONFIG["num_classes"],
    freeze_until_block=MODEL_CONFIG["freeze_until_block"],
    pretrained=MODEL_CONFIG["pretrained"],
    dropout=MODEL_CONFIG["dropout"],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"Freeze until block   : {MODEL_CONFIG['freeze_until_block']} / 16")


## 3. Training Stage

> ⏱️ Jauh lebih lambat per-epoch dibanding CNNsa-BiLSTM (model video, bukan keypoint). AMP aktif otomatis kalau ada GPU.

In [ ]:
from Training.MViTv2.Scripts.mvit_trainer import run_training

# Jalankan training loop (fine-tune)
run_training()


## 4. Evaluation Stage
Evaluasi model terbaik pada test split (video yang SAMA dengan test split CNNsa-BiLSTM).

In [ ]:
from Training.MViTv2.Scripts.mvit_evaluator import run_evaluation

# Jalankan evaluasi akhir pada test split -> tulis transformer_evaluation_summary.json
run_evaluation()


## 5. Visualisasi Hasil Training

In [ ]:
import json
import matplotlib.pyplot as plt
from Training.Config.shared_config import RESULTS_DIR

history_path = RESULTS_DIR / "mvit_v2_train_history.json"
if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.title("Loss Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, [x*100 for x in history["train_acc"]], label="Train Acc")
    plt.plot(epochs, [x*100 for x in history["val_acc"]], label="Val Acc")
    plt.title("Accuracy Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("History training belum tersedia. Silakan jalankan training terlebih dahulu.")


## 6. Tampilkan Confusion Matrix

In [ ]:
from IPython.display import Image, display
from Training.Config.shared_config import RESULTS_DIR

cm_path = RESULTS_DIR / "confusion_matrix_mvit_v2.png"
if cm_path.exists():
    display(Image(filename=str(cm_path)))
else:
    print("Confusion Matrix image belum dibuat.")


## 7. Bandingkan dengan CNNsa-BiLSTM
Jalankan setelah `Training/CNNsaBiLSTM/Train.ipynb` juga sudah dievaluasi (supaya
`evaluation_summary.json` proposed model tersedia).

In [ ]:
from Training.Scripts.comparator import compare_models

compare_models()
